# OpenAI Whisper API + pyannote Pipeline

## **Pipeline Overview**
This notebook creates a **cloud-based transcription pipeline** using OpenAI's Whisper API combined with local speaker diarization. We're testing this approach against our existing local models to evaluate:

### **Performance Comparison**
- **Speed**: API processing vs local GPU/CPU intensive models
- **Accuracy**: Latest OpenAI model vs open-source alternatives  
- **Resource Usage**: Cloud processing vs local memory/compute requirements
- **Cost**: API fees vs electricity/hardware costs

### **Expected Advantages**
- **Faster processing** - Cloud infrastructure vs local hardware
- **Latest model** - Most recent Whisper improvements not yet in open-source
- **No local resources** - No GPU memory limitations or long processing times
- **Consistent results** - Same performance regardless of local hardware

### **Pipeline Architecture**
```
Audio Input → OpenAI Whisper API → Transcript with Timestamps
     ↓
Speaker Diarization (pyannote) → Speaker-labeled Segments
     ↓
Integration Layer → Combined Transcript + Speaker IDs
     ↓
Analysis Ready → Emotion Detection, Bias Analysis, Political Insights
```

### **What We'll Build Next**
1. **Transcription Quality Comparison** - OpenAI vs WhisperX vs Local models
2. **Speed Benchmarking** - Processing time across different approaches
3. **Accuracy Metrics** - Word error rates and speaker identification precision
4. **Cost Analysis** - API costs vs computational expenses
5. **Integration with Emotion Analysis** - Seamless pipeline to bias detection
6. **Political Analysis Dashboard** - Speaker patterns, sentiment trends, bias indicators

This pipeline will serve as our **premium accuracy option** for critical analysis while maintaining our local alternatives for cost-sensitive or private processing needs.

---

In [1]:
# ============================================================
# SETUP - Import libraries and initialize components
# ============================================================

# Core libraries
import os
import json
import pandas as pd
import numpy as np
from datetime import datetime
import time

# OpenAI API setup
from dotenv import load_dotenv
from openai import OpenAI

# Display options for better pandas output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Load environment variables from .env file
load_dotenv()

# Initialize OpenAI client with API key
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Define audio file paths 
WAV_FILE = "../data/US_DebateAudio.wav"      # For pyannote diarization
MP3_FILE = "../data/US_DebateAudio.mp3"  # For Whisper API transcription

# Create outputs directory if it doesn't exist
os.makedirs("./outputs", exist_ok=True)

# Verify setup
api_key_loaded = bool(os.getenv("OPENAI_API_KEY"))
wav_exists = os.path.exists(WAV_FILE)
mp3_exists = os.path.exists(MP3_FILE)

print("=== SETUP COMPLETE ===")
print(f"API key loaded: {api_key_loaded}")
print(f"WAV file exists: {wav_exists} ({WAV_FILE})")
print(f"MP3 file exists: {mp3_exists} ({MP3_FILE})")
print(f"Outputs directory ready: {os.path.exists('./outputs')}")

if not all([api_key_loaded, wav_exists, mp3_exists]):
    print("Some required components are missing!")
else:
    print("Ready to proceed with pipeline!")

=== SETUP COMPLETE ===
API key loaded: True
WAV file exists: True (../data/US_DebateAudio.wav)
MP3 file exists: True (../data/US_DebateAudio.mp3)
Outputs directory ready: True
Ready to proceed with pipeline!


## Step 2 – Transcription with OpenAI Whisper API

We'll use OpenAI's Whisper API to transcribe our MP3 file and get **segments with timestamps**.

### **What this does:**
- Sends audio to OpenAI's cloud-based Whisper model
- Gets back transcript text + individual segments with start/end times
- Each segment is a phrase or sentence with precise timing
- This gives us the **"what was said"** part of our pipeline

### **API Settings:**
- **Model**: `whisper-1` (latest available via API)
- **Format**: `verbose_json` (includes timing and metadata)
- **Granularity**: `segment` level (phrases, not just words)

In [2]:
# ============================================================
# STEP 2: Transcribe audio with OpenAI Whisper API
# ============================================================

def transcribe_with_openai(audio_path):
    """
    Transcribe audio using OpenAI Whisper API
    
    Args:
        audio_path (str): Path to the audio file (MP3/WAV/etc.)
    
    Returns:
        OpenAI transcription response object with segments and text
    """
    print(f"Opening audio file: {audio_path}")
    
    # Check file size (OpenAI has 25MB limit)
    file_size_mb = os.path.getsize(audio_path) / (1024 * 1024)
    print(f"File size: {file_size_mb:.2f} MB")
    
    if file_size_mb > 25:
        print("WARNING: File exceeds 25MB limit - API call will fail")
        return None
    
    print("Sending to OpenAI Whisper API...")
    start_time = time.time()
    
    try:
        with open(audio_path, "rb") as audio_file:
            # Call OpenAI Whisper API with detailed response format
            response = client.audio.transcriptions.create(
                model="whisper-1",                          # Latest Whisper model
                file=audio_file,                           # Audio file object
                response_format="verbose_json",            # Get detailed response with metadata
                timestamp_granularities=["segment"]       # Get segment-level timestamps
            )
        
        elapsed_time = time.time() - start_time
        print(f"Transcription completed in {elapsed_time:.2f} seconds")
        
        return response
    
    except Exception as e:
        print(f"API Error: {e}")
        return None

# Run transcription on our MP3 file
print("Starting transcription with OpenAI Whisper API...")
whisper_result = transcribe_with_openai(MP3_FILE)

# Display results
if whisper_result:
    print("\n" + "="*60)
    print("TRANSCRIPTION RESULTS")
    print("="*60)
    
    # Basic info about the transcription
    print(f"Language detected: {getattr(whisper_result, 'language', 'unknown')}")
    print(f"Total duration: {getattr(whisper_result, 'duration', 0):.2f} seconds")
    print(f"Full text length: {len(whisper_result.text)} characters")
    
    # Show number of segments if available
    if hasattr(whisper_result, 'segments') and whisper_result.segments:
        print(f"Number of segments: {len(whisper_result.segments)}")
    else:
        print("No segment data returned")
    
    # Preview first 200 characters of transcript
    print(f"\nTranscript preview (first 200 chars):")
    print("-" * 40)
    preview_text = whisper_result.text[:200] + "..." if len(whisper_result.text) > 200 else whisper_result.text
    print(preview_text)
    
else:
    print("Transcription failed - cannot proceed to next step")

Starting transcription with OpenAI Whisper API...
Opening audio file: ../data/US_DebateAudio.mp3
File size: 10.87 MB
Sending to OpenAI Whisper API...
Transcription completed in 61.27 seconds

TRANSCRIPTION RESULTS
Language detected: english
Total duration: 565.77 seconds
Full text length: 8463 characters
Number of segments: 209

Transcript preview (first 200 chars):
----------------------------------------
She doesn't have a plan. She copied Biden's plan, and it's like four sentences, like run, spot, run, four sentences that are just, oh, we'll try and lower taxes. She doesn't have a plan. Take a look a...


## Step 3 – Convert Whisper segments to DataFrame

Now we'll extract the segment data from the OpenAI response and put it into a clean pandas DataFrame.

### **What this step does:**
- Extracts individual segments from `whisper_result.segments`
- Each segment has: start time, end time, and text content
- Creates a structured DataFrame we can easily work with
- This gives us the foundation for merging with speaker data

### **DataFrame structure:**
- `start_s`: When the segment starts (seconds)
- `end_s`: When the segment ends (seconds) 
- `text`: What was said in that time period

In [4]:
# ============================================================
# STEP 3: Convert Whisper segments to pandas DataFrame
# ============================================================

def extract_whisper_segments(whisper_response):
    """
    Extract segment data from OpenAI Whisper API response
    
    Args:
        whisper_response: Response object from OpenAI API
        
    Returns:
        pandas.DataFrame with columns: start_s, end_s, text
    """
    segments_list = []
    
    # Check if we have segments in the response
    if hasattr(whisper_response, 'segments') and whisper_response.segments:
        print(f"Processing {len(whisper_response.segments)} segments...")
        
        for i, segment in enumerate(whisper_response.segments):
            # Extract the key information from each segment
            segment_data = {
                'start_s': round(segment.start, 2),      # Start time in seconds
                'end_s': round(segment.end, 2),          # End time in seconds
                'text': segment.text.strip()             # Clean up the text
            }
            segments_list.append(segment_data)
            
        print(f"Extracted {len(segments_list)} segments successfully")
    else:
        print("o segments found in response")
    
    # Create DataFrame from the list of segment dictionaries
    df = pd.DataFrame(segments_list)
    return df

# Only proceed if we have a successful transcription
if whisper_result:
    print("Converting Whisper segments to DataFrame...")
    
    # Extract segments into a clean DataFrame
    whisper_df = extract_whisper_segments(whisper_result)
    
    # Display information about our DataFrame
    print(f"\n{'='*50}")
    print("WHISPER SEGMENTS DATAFRAME")
    print(f"{'='*50}")
    
    print(f"DataFrame shape: {whisper_df.shape}")
    print(f"Columns: {list(whisper_df.columns)}")
    
    if len(whisper_df) > 0:
        print(f"Time range: {whisper_df['start_s'].min():.1f}s to {whisper_df['end_s'].max():.1f}s")
        print(f"Total text length: {whisper_df['text'].str.len().sum()} characters")
        
        print(f"\nFirst 5 segments:")
        print(whisper_df.head())
        
        print(f"\nDataFrame info:")
        print(whisper_df.info())
    else:
        print("No segments to display")
        
else:
    print("Cannot proceed - no whisper_result from previous step")

Converting Whisper segments to DataFrame...
Processing 209 segments...
Extracted 209 segments successfully

WHISPER SEGMENTS DATAFRAME
DataFrame shape: (209, 3)
Columns: ['start_s', 'end_s', 'text']
Time range: 0.0s to 557.6s
Total text length: 8255 characters

First 5 segments:
   start_s  end_s                                               text
0     0.00   2.00                           She doesn't have a plan.
1     2.00   6.72  She copied Biden's plan, and it's like four se...
2     6.72  10.60  like run, spot, run, four sentences that are j...
3    10.60  12.76                     oh, we'll try and lower taxes.
4    12.76  13.68                           She doesn't have a plan.

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209 entries, 0 to 208
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   start_s  209 non-null    float64
 1   end_s    209 non-null    float64
 2   text     209 non-null    obj